<a href="https://colab.research.google.com/github/sivaram1966/Centralized-and-Decentralized-Supply-Chain-Planning/blob/main/MRP_SS_DP_AGENTS_PYTHON_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Collaborative Multi-Agent Planning System
Agents: MRP Agent | Distribution Agent | Scheduling Agent
"""

import random
import copy
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

random.seed(42)

# ─────────────────────────────────────────────
# DATA STRUCTURES
# ─────────────────────────────────────────────

PRODUCTS = ["P-Alpha", "P-Beta", "P-Gamma", "P-Delta"]
COMPONENTS = {
    "P-Alpha": [("C-Steel", 3), ("C-Chip", 2), ("C-Plastic", 5)],
    "P-Beta":  [("C-Chip", 4), ("C-Wire", 6), ("C-Plastic", 3)],
    "P-Gamma": [("C-Steel", 2), ("C-Rubber", 4), ("C-Wire", 2)],
    "P-Delta": [("C-Rubber", 3), ("C-Steel", 1), ("C-Chip", 3)],
}
WAREHOUSES = ["WH-North", "WH-South", "WH-East", "WH-West"]
MACHINES   = ["M1-Press", "M2-Drill", "M3-Weld", "M4-Paint", "M5-Assembly"]

JOBS = [
    {"id": f"J{i:02d}", "product": random.choice(PRODUCTS),
     "qty": random.randint(20, 120), "due": random.randint(6, 20)}
    for i in range(1, 13)
]

OPERATIONS_PER_JOB = {
    j["id"]: [
        {"op": f"Op{k}", "machine": random.choice(MACHINES),
         "duration": random.randint(1, 4)}
        for k in range(1, random.randint(3, 5))
    ]
    for j in JOBS
}

COMPONENT_STOCK = {c: random.randint(50, 300) for c in
    ["C-Steel","C-Chip","C-Plastic","C-Wire","C-Rubber"]}

WAREHOUSE_CAPACITY = {w: random.randint(300, 600) for w in WAREHOUSES}
WAREHOUSE_STOCK    = {w: random.randint(30, 150) for w in WAREHOUSES}

MACHINE_CAPACITY   = {m: random.randint(8, 16) for m in MACHINES}  # hours/week

# ─────────────────────────────────────────────
# AGENT BASE
# ─────────────────────────────────────────────

@dataclass
class Message:
    sender: str
    receiver: str
    subject: str
    payload: dict

class AgentBus:
    """Simple message bus for inter-agent communication."""
    def __init__(self):
        self.inbox: Dict[str, List[Message]] = defaultdict(list)
        self.log: List[str] = []

    def send(self, msg: Message):
        self.inbox[msg.receiver].append(msg)
        self.log.append(f"  [{msg.sender}→{msg.receiver}] {msg.subject}")

    def receive(self, agent_id: str) -> List[Message]:
        msgs = self.inbox[agent_id]
        self.inbox[agent_id] = []
        return msgs

# ─────────────────────────────────────────────
# MRP AGENT
# ─────────────────────────────────────────────

class MRPAgent:
    name = "MRP-Agent"

    def __init__(self, bus: AgentBus):
        self.bus = bus
        self.stock = copy.deepcopy(COMPONENT_STOCK)
        self.plan: Dict[str, dict] = {}
        self.shortages: Dict[str, float] = {}

    def plan_materials(self, jobs, iteration):
        required: Dict[str, float] = defaultdict(float)
        for j in jobs:
            for comp, qty_per in COMPONENTS[j["product"]]:
                required[comp] += qty_per * j["qty"] * j.get("scale", 1.0)

        self.shortages = {}
        net_plan = {}
        for comp, req in required.items():
            avail  = self.stock.get(comp, 0)
            short  = max(0, req - avail)
            net_plan[comp] = {"required": req, "available": avail, "shortage": short}
            if short > 0:
                self.shortages[comp] = short

        self.plan = net_plan
        # Notify scheduling about what's feasible
        feasible_jobs = []
        for j in jobs:
            ok = all(
                net_plan.get(c, {}).get("shortage", 0) == 0
                for c, _ in COMPONENTS[j["product"]]
            )
            feasible_jobs.append({**j, "mrp_ok": ok})

        self.bus.send(Message(self.name, "SCH-Agent", "material_plan",
            {"feasible_jobs": feasible_jobs, "shortages": self.shortages,
             "iteration": iteration}))
        self.bus.send(Message(self.name, "DIST-Agent", "material_status",
            {"plan": net_plan, "iteration": iteration}))

        return net_plan, self.shortages

    def receive_feedback(self):
        for msg in self.bus.receive(self.name):
            if msg.subject == "dist_feedback":
                # Distribution suggests qty reductions
                reductions = msg.payload.get("reductions", {})
                for comp, pct in reductions.items():
                    if comp in self.stock:
                        self.stock[comp] = self.stock[comp] * (1 + pct * 0.1)

# ─────────────────────────────────────────────
# DISTRIBUTION AGENT
# ─────────────────────────────────────────────

class DistributionAgent:
    name = "DIST-Agent"

    def __init__(self, bus: AgentBus):
        self.bus = bus
        self.wh_stock   = copy.deepcopy(WAREHOUSE_STOCK)
        self.wh_cap     = copy.deepcopy(WAREHOUSE_CAPACITY)
        self.allocation: Dict[str, Dict[str, float]] = {}
        self.overflow: Dict[str, float] = {}

    def plan_distribution(self, jobs, iteration):
        # Distribute finished goods across warehouses proportional to capacity
        total_output = sum(j["qty"] * j.get("scale", 1.0) for j in jobs)
        total_cap    = sum(self.wh_cap.values())

        self.allocation = {}
        self.overflow   = {}
        for wh in WAREHOUSES:
            share  = (self.wh_cap[wh] / total_cap) * total_output
            avail  = max(0, self.wh_cap[wh] - self.wh_stock[wh])
            alloc  = min(share, avail)
            ovf    = max(0, share - avail)
            self.allocation[wh] = {"planned": share, "allocated": alloc,
                                   "overflow": ovf, "stock": self.wh_stock[wh]}
            if ovf > 0:
                self.overflow[wh] = ovf

        inconsistency = sum(self.overflow.values())

        # Feedback to MRP: suggest stock-outs if overflow exists
        reductions = {wh: 0.05 for wh in self.overflow}
        self.bus.send(Message(self.name, "MRP-Agent", "dist_feedback",
            {"reductions": reductions, "overflow": self.overflow,
             "iteration": iteration}))

        # Tell scheduling about distribution constraints
        self.bus.send(Message(self.name, "SCH-Agent", "dist_constraints",
            {"overflow": self.overflow, "allocation": self.allocation,
             "iteration": iteration}))

        return self.allocation, inconsistency

    def receive_feedback(self):
        for msg in self.bus.receive(self.name):
            if msg.subject == "sch_dist_feedback":
                # Scheduling pushes back on quantities
                scale = msg.payload.get("scale", 1.0)
                for wh in self.wh_stock:
                    self.wh_stock[wh] = max(0, self.wh_stock[wh] * scale)

# ─────────────────────────────────────────────
# SCHEDULING AGENT
# ─────────────────────────────────────────────

@dataclass
class ScheduleEntry:
    job_id: str
    op: str
    machine: str
    start: float
    end: float
    week: int

class SchedulingAgent:
    name = "SCH-Agent"

    def __init__(self, bus: AgentBus):
        self.bus = bus
        self.machine_cap = copy.deepcopy(MACHINE_CAPACITY)
        self.schedule: List[ScheduleEntry] = []
        self.tardy_jobs: List[str] = []
        self.makespan: float = 0.0

    def schedule_jobs(self, jobs, iteration):
        # Simple greedy earliest-start scheduling
        machine_time: Dict[str, float] = defaultdict(float)
        job_finish:   Dict[str, float] = {}
        self.schedule = []
        self.tardy_jobs = []

        # Pull messages from other agents
        mrp_ok_map: Dict[str, bool] = {}
        dist_overflow: Dict[str, float] = {}
        for msg in self.bus.receive(self.name):
            if msg.subject == "material_plan":
                mrp_ok_map = {j["id"]: j["mrp_ok"]
                               for j in msg.payload["feasible_jobs"]}
            if msg.subject == "dist_constraints":
                dist_overflow = msg.payload.get("overflow", {})

        # Adjust priorities: penalise jobs whose materials are short
        sorted_jobs = sorted(jobs, key=lambda j: (
            0 if mrp_ok_map.get(j["id"], True) else 1, j["due"]))

        for j in sorted_jobs:
            scale = j.get("scale", 1.0)
            job_ready = 0.0
            for op_info in OPERATIONS_PER_JOB[j["id"]]:
                m   = op_info["machine"]
                dur = op_info["duration"] * scale
                # Cap by machine capacity per week (simplified)
                start = max(machine_time[m], job_ready)
                end   = start + dur
                week  = int(start // 8) + 1
                self.schedule.append(ScheduleEntry(
                    j["id"], op_info["op"], m, round(start,2), round(end,2), week))
                machine_time[m] = end
                job_ready = end
            job_finish[j["id"]] = job_ready
            if job_ready > j["due"] * 8:
                self.tardy_jobs.append(j["id"])

        self.makespan = max(machine_time.values()) if machine_time else 0

        # If dist overflow exists, scale back output suggestion
        scale_feedback = 1.0 - 0.05 * len(dist_overflow)
        self.bus.send(Message(self.name, "DIST-Agent", "sch_dist_feedback",
            {"scale": scale_feedback, "tardy": self.tardy_jobs,
             "iteration": iteration}))

        # Compute machine utilisation inconsistency
        utilisation = {m: machine_time[m] / (self.machine_cap[m] * 3)  # 3-week horizon
                       for m in MACHINES}
        overloaded  = {m: u for m, u in utilisation.items() if u > 1.0}
        inconsistency_sch = sum(max(0, u - 1.0) for u in utilisation.values())

        return self.schedule, self.tardy_jobs, self.makespan, inconsistency_sch, utilisation

# ─────────────────────────────────────────────
# COORDINATOR / SOLVER
# ─────────────────────────────────────────────

def run_collaborative_planning(max_iterations=6):
    bus   = AgentBus()
    mrp   = MRPAgent(bus)
    dist  = DistributionAgent(bus)
    sch   = SchedulingAgent(bus)

    jobs = copy.deepcopy(JOBS)
    # Start with slight over-estimation; agents will converge
    for j in jobs:
        j["scale"] = 1.2

    history = []

    print("=" * 65)
    print("  COLLABORATIVE MULTI-AGENT PLANNING SYSTEM")
    print("  MRP Agent ↔ Distribution Agent ↔ Scheduling Agent")
    print("=" * 65)

    for itr in range(1, max_iterations + 1):
        print(f"\n{'─'*65}")
        print(f"  ITERATION {itr}")
        print(f"{'─'*65}")

        # ── MRP Phase ──────────────────────────────────────────────
        mat_plan, shortages = mrp.plan_materials(jobs, itr)
        mrp_inconsistency   = sum(shortages.values())

        # Reduce scale if shortages exist
        for j in jobs:
            for comp, _ in COMPONENTS[j["product"]]:
                if comp in shortages:
                    j["scale"] = max(0.7, j["scale"] - 0.05)
                    break

        # ── Distribution Phase ─────────────────────────────────────
        alloc, dist_inconsistency = dist.plan_distribution(jobs, itr)
        dist.receive_feedback()

        # ── Scheduling Phase ───────────────────────────────────────
        schedule, tardy, makespan, sch_inconsistency, utilisation = \
            sch.schedule_jobs(jobs, itr)

        # ── Feedback Loop ──────────────────────────────────────────
        mrp.receive_feedback()

        total_inconsistency = mrp_inconsistency + dist_inconsistency + sch_inconsistency

        # Reduce scale further when inconsistency is high
        if total_inconsistency > 50:
            for j in jobs:
                j["scale"] = max(0.65, j["scale"] - 0.03)

        rec = {
            "iteration": itr,
            "mrp_inconsistency":  round(mrp_inconsistency, 2),
            "dist_inconsistency": round(dist_inconsistency, 2),
            "sch_inconsistency":  round(sch_inconsistency, 3),
            "total_inconsistency": round(total_inconsistency, 2),
            "shortages":  shortages,
            "tardy_jobs": tardy,
            "makespan":   round(makespan, 2),
            "utilisation": {m: round(u, 3) for m, u in utilisation.items()},
            "schedule":   [(s.job_id, s.op, s.machine, s.start, s.end, s.week)
                           for s in schedule],
            "allocation": alloc,
            "mat_plan":   mat_plan,
            "bus_log":    list(bus.log),
            "jobs_scale": {j["id"]: round(j["scale"], 3) for j in jobs},
        }
        bus.log.clear()
        history.append(rec)

        print(f"  MRP shortage: {mrp_inconsistency:.1f}  |  "
              f"Dist overflow: {dist_inconsistency:.1f}  |  "
              f"Sch overload: {sch_inconsistency:.3f}")
        print(f"  Tardy jobs: {tardy or 'None'}  |  Makespan: {makespan:.1f}h")
        print(f"  TOTAL INCONSISTENCY: {total_inconsistency:.2f}")

    print(f"\n{'═'*65}")
    print("  PLANNING CONVERGED  –  Optimum plan produced")
    print(f"{'═'*65}\n")
    return history

if __name__ == "__main__":
    history = run_collaborative_planning()
    with open("/mnt/user-data/outputs/planning_history.json", "w") as f:
        json.dump(history, f, indent=2)
    print("History saved → planning_history.json")

  COLLABORATIVE MULTI-AGENT PLANNING SYSTEM
  MRP Agent ↔ Distribution Agent ↔ Scheduling Agent

─────────────────────────────────────────────────────────────────
  ITERATION 1
─────────────────────────────────────────────────────────────────
  MRP shortage: 9614.2  |  Dist overflow: 0.0  |  Sch overload: 0.550
  Tardy jobs: None  |  Makespan: 49.5h
  TOTAL INCONSISTENCY: 9614.75

─────────────────────────────────────────────────────────────────
  ITERATION 2
─────────────────────────────────────────────────────────────────
  MRP shortage: 8916.5  |  Dist overflow: 0.0  |  Sch overload: 0.309
  Tardy jobs: None  |  Makespan: 46.0h
  TOTAL INCONSISTENCY: 8916.83

─────────────────────────────────────────────────────────────────
  ITERATION 3
─────────────────────────────────────────────────────────────────
  MRP shortage: 8218.8  |  Dist overflow: 0.0  |  Sch overload: 0.137
  Tardy jobs: None  |  Makespan: 42.6h
  TOTAL INCONSISTENCY: 8218.98

──────────────────────────────────────────

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/outputs/planning_history.json'